In [1]:
import math
import os.path
import numpy as np
import pandas as pd

from data_management import TrainValidationSplitsIdxs, save_train_validation_splits, load_train_validation_splits, \
    ActivationsDataSelector
from phi_3_5_constants import seed, dsets_index_path

In [2]:
num_splits = 10#TODO rerun this script with 20 instead of 10, to match Burger et al

seeds = [int((seed+739*k)*(math.pi-2)**k) for k in range(num_splits)]
np_rngs = [np.random.default_rng(seed) for seed in seeds]

train_fraction = 0.8

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")

In [4]:
#dict of dset idx (in dsets_index_df) to list of record indices 
train_valid_splits_spec: dict[int, list[TrainValidationSplitsIdxs]] = {}

In [5]:
for dset_idx, row in dsets_index_df.iterrows():
    categ_nm = row["Categ_Folder"]
    dset_file_nm = row["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dset_size = row["max_global_record_idx"] - row["min_global_record_idx"]
    train_size = math.floor(dset_size*train_fraction)
    
    curr_dset_splits: list[TrainValidationSplitsIdxs] = []
    
    for split_variant_idx in range(num_splits):
        train_split_indices = sorted(np_rngs[split_variant_idx].choice(dset_size, train_size, replace=False).tolist())
        validation_split_indices = sorted(list(set(range(dset_size)) - set(train_split_indices)))
        curr_dset_splits.append(TrainValidationSplitsIdxs(train_split_indices, validation_split_indices))
    
    train_valid_splits_spec[dset_idx] = curr_dset_splits

In [6]:
save_train_validation_splits(train_valid_splits_spec)

In [7]:
#remainder of script is just for testing runtime type checking
reloaded_train_valid_splits_spec = load_train_validation_splits() 

In [8]:
selector = ActivationsDataSelector(dsets_index_df, reloaded_train_valid_splits_spec)

In [9]:
selector.select_train_validation_for_scenario([0], 4)

BeartypeDoorHintViolation: Die_if_unbearable() value "tensor([[1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
    ...]])" violates type hint <class 'jaxtyping.Float[Tensor, 'combined_t_dset_sz 5']'>, as the dimension size 1 does not equal 5 as expected by the type hint.